# Caso D · 05 Validación IAQ y alertas según OMS / EN 16798

> _Tutorial · Caso de uso: **D — IAQ + ocupación** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Implementar un agregador IAQ que dispara alertas cuando los valores salen de los rangos recomendados por OMS / EN 16798.


## 2. Qué se aprende

- Mapear rangos a categorías.
- Generar alertas por minuto.
- Visualizar tiempos en cada categoría.


## 3. Contexto del caso de uso

El profesor recibe alertas cuando CO₂ > 1500 ppm o IAQ > 200 — señal de ventilar.


## 4. Relación con CENTINELA+

Las alertas van a `telemetry_events` o se exponen vía API REST.


## 5. Relación con Medallion

Oro: regla + reporte.


## 6. Datos de entrada

Mock In-Gauge.


## 7. Schema CAPTIA esperado

No aplica.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Reusamos In-Gauge.


In [ ]:
csv_path = ROOT / "notebooks" / "_data" / "ingauge_aula01_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"]).set_index("timestamp")
df.head()


## 10. Exploración paso a paso

Categorización CO₂.


In [ ]:
def cat_co2(x):
    if x < 800: return "óptimo"
    if x < 1000: return "aceptable"
    if x < 1500: return "vigilar"
    if x < 2000: return "molesto"
    return "ventilar"

df["co2_cat"] = df["Indoor_CO2"].apply(cat_co2)
print(df["co2_cat"].value_counts(normalize=True).round(3))


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Generamos alertas.


In [ ]:
alertas = df[df["Indoor_CO2"] > 1500].copy()
alertas["msg"] = alertas["Indoor_CO2"].apply(lambda v: f"CO2={int(v)} ppm — abrir ventanas")
print(alertas[["msg"]].head())
print(f"Total alertas: {len(alertas)}")


## 13. Visualizaciones explicativas

Heatmap horario de la categoría más frecuente.


In [ ]:
df["hour"] = df.index.hour
heat = (df.groupby("hour")["co2_cat"]
          .value_counts(normalize=True)
          .unstack(fill_value=0))
heat.plot.bar(stacked=True, figsize=(10, 3), colormap="viridis")
plt.title("Categoría CO2 por hora")
plt.tight_layout()


## 14. Validaciones

Confirmamos que ningún valor en horario lectivo cae en `extremo` con el mock (debería ser raro).


In [ ]:
mask = (df.index.hour.isin(range(8, 15))) & (df.index.dayofweek < 5)
extremo = (df.loc[mask, "Indoor_CO2"] > 4000).sum()
print(f"Extremo en lectivo: {extremo} puntos")


## 15. Errores comunes

1. **Threshold único**: usar tabla con varias categorías.
2. **Olvidar histeresis**: alertas oscilantes.
3. **Comparar contra exterior**: la regla EN 16798 lo recomienda.


## 16. Ejercicios propuestos

1. Añade categorías para temperatura.
2. Implementa histeresis (alerta solo si > 1500 durante > 5 min).
3. Crea una regla compuesta CO₂ + ruido.


## 17. Cómo se reutiliza con datos reales

Mismas reglas se aplican vía Flux task; ya tenemos un esqueleto.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `05_case_E_weather_solar/01_eda_era5.ipynb`.
- Documento web del caso: `docs/use-cases/case-d-iaq-occupancy.md`.
